# XOPT - OR-LIBRARY p-MEDIAN BENCHMARK

This notebook runs the `pymedian` metaheuristic on OR-Library p-median instances, stores one row per execution, aggregates the benchmark metrics by instance, and prints publication-ready LaTeX tables.

### PROJECT SETUP

In [1]:
import re
import sys
import numpy  as np
import pandas as pd

from pathlib         import Path
from tqdm.auto       import tqdm
from IPython.display import display
from time            import perf_counter

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / 'instances').exists() and \
           (candidate / 'pymedian' ).exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root containing 'instances' and 'pymedian'."
    )


def instance_sort_key(pathlike: str | Path) -> tuple[int, str]:
    stem  = Path     (pathlike ).stem
    match = re.search(r'(\d+)$', stem)

    if match is None:
        return (10**9, stem)

    return (int(match.group(1)), stem)

In [3]:
PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pymedian

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

PROJECT_ROOT = /home/rei-luisinho/xopt


In [4]:
def canonical_instance_name(instance_name: str) -> str:
    instance_name = instance_name.strip()

    if instance_name.endswith('.txt'):
        return instance_name

    return f'{instance_name}.txt'


def list_orlibrary_instances(instances_dir: Path) -> list[str]:
    return sorted(
        [
            path.name
            for path in instances_dir.glob('pmed*.txt')
            if  path.name != 'pmedopt.txt'
        ],
        key=instance_sort_key,
    )


def read_instance_metadata(instance_path: Path) -> dict[str, int]:
    header = instance_path.read_text().splitlines()[0].split()

    if len(header) < 3:
        raise ValueError(f'Could not parse instance header: {instance_path}')

    return {
        'n': int(header[0]),
        'p': int(header[2]),
    }


def load_best_known_costs(pmedopt_path: Path) -> pd.DataFrame:
    rows = []

    for raw_line in pmedopt_path.read_text().splitlines()[1:]:
        line = raw_line.strip()
        if not line:
            continue

        parts = line.split()

        rows.append(
            {
                'instance_id'     : parts[0].strip(),
                'best_known_cost' : float(parts[1] ),
            }
        )

    df = pd.DataFrame(rows)

    df['instance_order'] = df['instance_id'].map(
        lambda value: instance_sort_key(value)[0]
    )

    return df.sort_values(['instance_order', 'instance_id']) \
             .drop       (columns='instance_order'         ) \
             .reset_index(drop   =True)


def compute_gap_percent(cost: float | None, best_known_cost: float | None) -> float:
    if cost is None or pd.isna(cost):
        return np.nan
    if best_known_cost is None  or \
       pd.isna(best_known_cost) or \
       float  (best_known_cost) == 0.0:
        return np.nan

    return 100.0 * (float(cost) - float(best_known_cost)) / float(best_known_cost)

### EXPERIMENT CONFIGURATION

Adjust the instance list, number of runs, and main metaheuristic parameters here.

In [5]:
INSTANCES_DIR = PROJECT_ROOT  / 'instances'
PMEDOPT_PATH  = INSTANCES_DIR / 'pmedopt.txt'

DEFAULT_INSTANCE_NAMES = list_orlibrary_instances(INSTANCES_DIR)
INSTANCE_NAMES         = DEFAULT_INSTANCE_NAMES
N_RUNS                 = 1


RESTARTS       = 8
MAX_ITER       = 25
FACTOR         = 1
DETAILS_FORMAT = 'indices'


TIME_LIMIT_SECONDS = None

SAVE_RAW_CSV = True
SAVE_AGG_CSV = True

OUTPUT_DIR      = PROJECT_ROOT / 'notebooks' / 'experiments_sbpo' / 'artifacts'
RAW_RESULTS_CSV = OUTPUT_DIR   / 'pmedian_benchmark_raw.csv'
AGG_RESULTS_CSV = OUTPUT_DIR   / 'pmedian_benchmark_aggregated.csv'

BEST_KNOWN_COSTS_DF = load_best_known_costs(PMEDOPT_PATH)
BEST_KNOWN_COSTS    = BEST_KNOWN_COSTS_DF.set_index('instance_id')['best_known_cost'].to_dict()

In [6]:
print(f'Instances folder     : {INSTANCES_DIR      }')
print(f'Number of instances  : {len(INSTANCE_NAMES)}')
print(f'Runs per instance    : {N_RUNS             }')
print(f'Time limit parameter : {TIME_LIMIT_SECONDS }')

print()
print(f'Metaheuristic parameters : restarts={RESTARTS}, max_iter={MAX_ITER}, factor={FACTOR}')

Instances folder     : /home/rei-luisinho/xopt/instances
Number of instances  : 40
Runs per instance    : 1
Time limit parameter : None

Metaheuristic parameters : restarts=8, max_iter=25, factor=1


In [7]:
display(BEST_KNOWN_COSTS_DF.head())

,instance_id,best_known_cost
0,pmed1,5819.0
1,pmed2,4093.0
2,pmed3,4250.0
3,pmed4,3034.0
4,pmed5,1355.0


> Notes on available metrics:
>
> `instances/pmedopt.txt` provides best known costs, not best known solutions;
>
> The public long-term memory returned by `details['long_term_memory']` is already distinct, so `memory_size` and `distinct_memory_size` coincide with the current API;
>
> The C++ backend is stochastic and currently uses internal random initialization, so repeated runs are meaningful for benchmark statistics.

### LOAD BENCHMARK INSTANCES

In [8]:
selected_instances_df = pd.DataFrame(
    [
        {
            'instance'        : canonical_instance_name     (instance_name)      ,
            'instance_id'     : Path(canonical_instance_name(instance_name)).stem,
            'best_known_cost' : BEST_KNOWN_COSTS.get(
                Path(
                    canonical_instance_name(instance_name)
                ).stem, np.nan
            ),

            **read_instance_metadata(INSTANCES_DIR / canonical_instance_name(instance_name)),
        }
        for instance_name in INSTANCE_NAMES
    ]
)

selected_instances_df['instance_order'] = selected_instances_df['instance'].map(
    lambda value: instance_sort_key(value)[0]
)

selected_instances_df = selected_instances_df.sort_values(['instance_order', 'instance']) \
                                             .drop       (columns='instance_order'      ) \
                                             .reset_index(drop   =True)


print(f'Selected instances: {len(selected_instances_df)}.')

Selected instances: 40.


In [9]:
display(selected_instances_df.head(10))

,instance,instance_id,best_known_cost,n,p
0,pmed1.txt,pmed1,5819.0,100,5
1,pmed2.txt,pmed2,4093.0,100,10
2,pmed3.txt,pmed3,4250.0,100,10
3,pmed4.txt,pmed4,3034.0,100,20
4,pmed5.txt,pmed5,1355.0,100,33
5,pmed6.txt,pmed6,7824.0,200,5
6,pmed7.txt,pmed7,5631.0,200,10
7,pmed8.txt,pmed8,4445.0,200,20
8,pmed9.txt,pmed9,2734.0,200,40
9,pmed10.txt,pmed10,1255.0,200,67


### FUNCTION TO RUN A SINGLE ROUND

The function returns one dictionary per execution and never raises for instance-level failures:

In [10]:
def run_single_round(
    instance_name: str,
    *,
    restarts          : int,
    max_iter          : int,
    factor            : int,
    details_format    : str = 'indices'    ,
    time_limit_seconds: float | None = None,
) -> dict[str, object]:
    instance_name = canonical_instance_name(instance_name)
    instance_path = INSTANCES_DIR / instance_name
    instance_id   = Path(instance_name).stem

    row = {
        'instance'             : instance_name,
        'instance_id'          : instance_id,
        'runtime_seconds'      : np.nan,
        'best_cost'            : np.nan,
        'best_solution'        : None  ,
        'memory_size'          : np.nan,
        'n'                    : np.nan,
        'p'                    : np.nan,
        'gap_percent'          : np.nan,
        'best_known_cost'      : BEST_KNOWN_COSTS.get(instance_id, np.nan),
        'status'               : 'error'           ,
        'error_message'        : None              ,
    }

    started_at = perf_counter()

    if not instance_path.exists():
        row['runtime_seconds'] = perf_counter() - started_at
        row['error_message'  ] = f'Instance not found: {instance_path}'

        return row

    metadata = read_instance_metadata(instance_path)
    row['n'] = metadata['n']
    row['p'] = metadata['p']

    try:
        summary, details = pymedian.solve_pmedian(
            instance_path,
            restarts      =restarts      ,
            max_iter      =max_iter      ,
            factor        =factor        ,
            details_format=details_format,
        )

        runtime_seconds  = perf_counter() - started_at

        long_term_memory = details    .get('long_term_memory', [])
        memory_size      = int(summary.get('long_term_mem'   , len(long_term_memory)))

        best_cost     = float(summary['tspmed_cost'])
        best_solution = [int(value) for value in summary['tspmed_facilities']]

        row.update(
            {
                'runtime_seconds'     : runtime_seconds     ,
                'best_cost'           : best_cost           ,
                'best_solution'       : best_solution       ,
                'memory_size'         : memory_size         ,
                'gap_percent'         : compute_gap_percent(best_cost, row['best_known_cost']),

                'n'                   : int(summary['n']),
                'p'                   : int(summary['p']),

                'status'              : 'ok',
                'error_message'       : None,
            }
        )
    except Exception as exc:
        row['runtime_seconds'] = perf_counter() - started_at
        row['error_message'  ] = f'{type(exc).__name__}: {exc}'

    return row


def run_benchmark(
    instance_names: list[str],
    *,
    n_runs            : int,
    restarts          : int,
    max_iter          : int,
    factor            : int,
    details_format    : str = 'indices'    ,
    time_limit_seconds: float | None = None,
) -> pd.DataFrame:
    if not instance_names:
        raise ValueError('The benchmark requires at least one instance.')

    jobs = [
        (
            canonical_instance_name(instance_name), run_id
        )
        for instance_name in instance_names
        for run_id        in range(1, n_runs + 1)
    ]

    rows = []

    for instance_name, run_id in tqdm(jobs,
                                      total        =len(jobs)       ,
                                      desc         ='Benchmark runs',
                                      dynamic_ncols=True            ):
        row = run_single_round(
            instance_name,
            restarts          =restarts          ,
            max_iter          =max_iter          ,
            factor            =factor            ,
            details_format    =details_format    ,
            time_limit_seconds=time_limit_seconds,
        )
        row['run_id'] = run_id

        rows.append(row)

    raw_df = pd.DataFrame(rows)

    raw_df['instance_order'] = raw_df['instance'].map(
        lambda value: instance_sort_key(value)[0]
    )

    return raw_df.sort_values(['instance_order', 'instance', 'run_id']) \
                 .drop       (columns='instance_order'                ) \
                 .reset_index(drop   =True)

### BATCH EXECUTION

In [11]:
raw_results_df = run_benchmark(
    INSTANCE_NAMES,
    n_runs            =N_RUNS            ,
    restarts          =RESTARTS          ,
    max_iter          =MAX_ITER          ,
    factor            =FACTOR            ,
    details_format    =DETAILS_FORMAT    ,
    time_limit_seconds=TIME_LIMIT_SECONDS,
)


print(f'Total executions : {len(raw_results_df)}')
print(f'Successful runs  : {(raw_results_df["status"] == "ok").sum()}')
print(f'Failed runs      : {(raw_results_df["status"] != "ok").sum()}')

Benchmark runs:   0%|          | 0/40 [00:00<?, ?it/s]

Total executions : 40
Successful runs  : 40
Failed runs      : 0


In [12]:
display(raw_results_df.head())

,instance,instance_id,runtime_seconds,best_cost,best_solution,memory_size,n,p,gap_percent,best_known_cost,status,error_message,run_id
0,pmed1.txt,pmed1,0.032671,5819.0,"[13, 99, 7, 91, 65]",127,100,5,0.0,5819.0,ok,None,1
1,pmed2.txt,pmed2,0.064219,4093.0,"[91, 37, 67, 8, 45, 99, 12, 95, 6, 41]",89,100,10,0.0,4093.0,ok,None,1
2,pmed3.txt,pmed3,0.055950,4250.0,"[9, 36, 48, 99, 13, 21, 26, 5, 69, 55]",103,100,10,0.0,4250.0,ok,None,1
3,pmed4.txt,pmed4,0.101170,3034.0,"[9, 8, 77, 55, 34, 5, 26, 38, 13, 22, 93, 87, ...",102,100,20,0.0,3034.0,ok,None,1
4,pmed5.txt,pmed5,0.116804,1355.0,"[94, 41, 84, 55, 9, 38, 75, 4, 1, 49, 28, 70, ...",93,100,33,0.0,1355.0,ok,None,1


### CONSOLIDATE RAW RESULTS

Persist the execution-level dataframe if desired and inspect any failures without stopping the notebook:

In [13]:
failure_df = raw_results_df.loc[
    raw_results_df['status'] != 'ok',
    [
        'instance'     ,
        'run_id'       ,
        'error_message',
    ]
].reset_index(drop=True)


if SAVE_RAW_CSV:
    OUTPUT_DIR    .mkdir (parents=True, exist_ok=True )
    raw_results_df.to_csv(RAW_RESULTS_CSV, index=False)

    print(f'Raw results saved to: {RAW_RESULTS_CSV}')

if failure_df.empty:
    print('No execution failures were recorded.')

Raw results saved to: /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/pmedian_benchmark_raw.csv
No execution failures were recorded.


In [14]:
display(raw_results_df.head(10))

,instance,instance_id,runtime_seconds,best_cost,best_solution,memory_size,n,p,gap_percent,best_known_cost,status,error_message,run_id
0,pmed1.txt,pmed1,0.032671,5819.0,"[13, 99, 7, 91, 65]",127,100,5,0.0,5819.0,ok,None,1
1,pmed2.txt,pmed2,0.064219,4093.0,"[91, 37, 67, 8, 45, 99, 12, 95, 6, 41]",89,100,10,0.0,4093.0,ok,None,1
2,pmed3.txt,pmed3,0.055950,4250.0,"[9, 36, 48, 99, 13, 21, 26, 5, 69, 55]",103,100,10,0.0,4250.0,ok,None,1
3,pmed4.txt,pmed4,0.101170,3034.0,"[9, 8, 77, 55, 34, 5, 26, 38, 13, 22, 93, 87, ...",102,100,20,0.0,3034.0,ok,None,1
4,pmed5.txt,pmed5,0.116804,1355.0,"[94, 41, 84, 55, 9, 38, 75, 4, 1, 49, 28, 70, ...",93,100,33,0.0,1355.0,ok,None,1
5,pmed6.txt,pmed6,0.186795,7824.0,"[111, 86, 101, 126, 16]",246,200,5,0.0,7824.0,ok,None,1
6,pmed7.txt,pmed7,0.343782,5631.0,"[181, 142, 10, 191, 186, 72, 3, 87, 131, 116]",135,200,10,0.0,5631.0,ok,None,1
7,pmed8.txt,pmed8,0.763342,4445.0,"[96, 130, 179, 76, 66, 194, 154, 104, 167, 70,...",90,200,20,0.0,4445.0,ok,None,1
8,pmed9.txt,pmed9,0.985864,2734.0,"[122, 25, 109, 19, 155, 125, 81, 31, 89, 77, 7...",130,200,40,0.0,2734.0,ok,None,1
9,pmed10.txt,pmed10,1.772026,1255.0,"[85, 104, 39, 47, 68, 158, 196, 124, 112, 180,...",177,200,67,0.0,1255.0,ok,None,1


### AGGREGATE RESULTS BY INSTANCE

The aggregated dataframe keeps only successful runs and computes the publication metrics requested for the benchmark table.

In [15]:
def aggregate_benchmark_results(
    raw_df: pd.DataFrame
) -> pd.DataFrame:
    ok_df = raw_df.loc[raw_df['status'] == 'ok'].copy()

    if ok_df.empty:
        raise ValueError(
            'No successful runs are available for aggregation.'
        )

    agg_df = (
        ok_df
        .groupby(
            [
                'instance'       ,
                'instance_id'    ,
                'n'              ,
                'p'              ,
                'best_known_cost',
            ],
            dropna=False
        )
        .agg    (
            runs                     =('run_id'               , 'size'),
            best_cost_obtained       =('best_cost'            , 'min' ),
            mean_cost                =('best_cost'            , 'mean'),
            mean_runtime_seconds     =('runtime_seconds'      , 'mean'),
            mean_memory_size         =('memory_size'          , 'mean'),
            std_cost                 =('best_cost'            , lambda s: float(s.std(ddof=0))),
            std_runtime_seconds      =('runtime_seconds'      , lambda s: float(s.std(ddof=0))),
            std_memory_size          =('memory_size'          , lambda s: float(s.std(ddof=0))),
        )
        .reset_index()
    )

    failure_counts = (
        raw_df
        .assign (
            failed=raw_df['status'] != 'ok'
        )
        .groupby(
            [
                'instance'   ,
                'instance_id',
            ],
            as_index=False
        )
        .agg(
            failed_runs=('failed', 'sum')
        )
    )

    agg_df = agg_df.merge(
        failure_counts,
        on =[
            'instance'   ,
            'instance_id',
        ],
        how='left'
    )

    agg_df['best_gap_percent'] = agg_df.apply(lambda row: compute_gap_percent(row['best_cost_obtained'], row['best_known_cost']), axis=1)
    agg_df['mean_gap_percent'] = agg_df.apply(lambda row: compute_gap_percent(row['mean_cost'         ], row['best_known_cost']), axis=1)

    agg_df['instance_order'  ] = agg_df['instance'].map(lambda value: instance_sort_key(value)[0])

    return agg_df.sort_values(['instance_order', 'instance']) \
                 .drop       (columns='instance_order'      ) \
                 .reset_index(drop   =True)


aggregated_results_df = aggregate_benchmark_results(raw_results_df)


if SAVE_AGG_CSV:
    OUTPUT_DIR           .mkdir (parents=True, exist_ok=True )
    aggregated_results_df.to_csv(AGG_RESULTS_CSV, index=False)

    print(f'Aggregated results saved to: {AGG_RESULTS_CSV}')

Aggregated results saved to: /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/pmedian_benchmark_aggregated.csv


In [16]:
display(aggregated_results_df.head(10))

,instance,instance_id,n,p,best_known_cost,runs,best_cost_obtained,mean_cost,mean_runtime_seconds,mean_memory_size,std_cost,std_runtime_seconds,std_memory_size,failed_runs,best_gap_percent,mean_gap_percent
0,pmed1.txt,pmed1,100,5,5819.0,1,5819.0,5819.0,0.032671,127.0,0.0,0.0,0.0,0,0.0,0.0
1,pmed2.txt,pmed2,100,10,4093.0,1,4093.0,4093.0,0.064219,89.0,0.0,0.0,0.0,0,0.0,0.0
2,pmed3.txt,pmed3,100,10,4250.0,1,4250.0,4250.0,0.055950,103.0,0.0,0.0,0.0,0,0.0,0.0
3,pmed4.txt,pmed4,100,20,3034.0,1,3034.0,3034.0,0.101170,102.0,0.0,0.0,0.0,0,0.0,0.0
4,pmed5.txt,pmed5,100,33,1355.0,1,1355.0,1355.0,0.116804,93.0,0.0,0.0,0.0,0,0.0,0.0
5,pmed6.txt,pmed6,200,5,7824.0,1,7824.0,7824.0,0.186795,246.0,0.0,0.0,0.0,0,0.0,0.0
6,pmed7.txt,pmed7,200,10,5631.0,1,5631.0,5631.0,0.343782,135.0,0.0,0.0,0.0,0,0.0,0.0
7,pmed8.txt,pmed8,200,20,4445.0,1,4445.0,4445.0,0.763342,90.0,0.0,0.0,0.0,0,0.0,0.0
8,pmed9.txt,pmed9,200,40,2734.0,1,2734.0,2734.0,0.985864,130.0,0.0,0.0,0.0,0,0.0,0.0
9,pmed10.txt,pmed10,200,67,1255.0,1,1255.0,1255.0,1.772026,177.0,0.0,0.0,0.0,0,0.0,0.0


### EXPORT PUBLICATION TABLES TO LATEX

In [17]:
def format_publication_value(
    value: float | int | None,
    *,
    digits : int  = 2    ,
    integer: bool = False,
) -> str:
    if value is None or pd.isna(value):
        return '--'

    if integer:
        return f'{int(round(float(value)))}'

    return f'{float(value):.{digits}f}'


def build_publication_tables(
    agg_df: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    full_table_df = pd.DataFrame(
        {
            'Instance'    : agg_df['instance_id'],
            'n'           : agg_df['n'                    ].map(lambda value: format_publication_value(value, integer=True)),
            'p'           : agg_df['p'                    ].map(lambda value: format_publication_value(value, integer=True)),
            'Best'        : agg_df['best_cost_obtained'   ].map(lambda value: format_publication_value(value, digits=0)),
            'Mean'        : agg_df['mean_cost'            ].map(lambda value: format_publication_value(value, digits=2)),
            'Std'         : agg_df['std_cost'             ].map(lambda value: format_publication_value(value, digits=2)),
            'BKS'         : agg_df['best_known_cost'      ].map(lambda value: format_publication_value(value, digits=0)),
            'BestGap(\\%)': agg_df['best_gap_percent'     ].map(lambda value: format_publication_value(value, digits=2)),
            'MeanGap(\\%)': agg_df['mean_gap_percent'     ].map(lambda value: format_publication_value(value, digits=2)),
            'Time(s)'     : agg_df['mean_runtime_seconds' ].map(lambda value: format_publication_value(value, digits=4)),
            'TimeStd'     : agg_df['std_runtime_seconds'  ].map(lambda value: format_publication_value(value, digits=4)),
            'Mem'         : agg_df['mean_memory_size'     ].map(lambda value: format_publication_value(value, digits=2)),
            'MemStd'      : agg_df['std_memory_size'      ].map(lambda value: format_publication_value(value, digits=2)),
        }
    )

    summary_table_df = full_table_df[
        [
            'Instance'    ,
            'n'           ,
            'p'           ,
            'Best'        ,
            'BKS'         ,
            'BestGap(\\%)',
            'MeanGap(\\%)',
            'Time(s)'     ,
            'Mem'         ,
        ]
    ].copy()

    return full_table_df, summary_table_df


publication_full_df, publication_summary_df = build_publication_tables(aggregated_results_df)

display(publication_full_df   .head(10))
display(publication_summary_df.head(10))

,Instance,n,p,Best,Mean,Std,BKS,BestGap(\%),MeanGap(\%),Time(s),TimeStd,Mem,MemStd
0,pmed1,100,5,5819,5819.00,0.00,5819,0.00,0.00,0.0327,0.0000,127.00,0.00
1,pmed2,100,10,4093,4093.00,0.00,4093,0.00,0.00,0.0642,0.0000,89.00,0.00
2,pmed3,100,10,4250,4250.00,0.00,4250,0.00,0.00,0.0560,0.0000,103.00,0.00
3,pmed4,100,20,3034,3034.00,0.00,3034,0.00,0.00,0.1012,0.0000,102.00,0.00
4,pmed5,100,33,1355,1355.00,0.00,1355,0.00,0.00,0.1168,0.0000,93.00,0.00
5,pmed6,200,5,7824,7824.00,0.00,7824,0.00,0.00,0.1868,0.0000,246.00,0.00
6,pmed7,200,10,5631,5631.00,0.00,5631,0.00,0.00,0.3438,0.0000,135.00,0.00
7,pmed8,200,20,4445,4445.00,0.00,4445,0.00,0.00,0.7633,0.0000,90.00,0.00
8,pmed9,200,40,2734,2734.00,0.00,2734,0.00,0.00,0.9859,0.0000,130.00,0.00
9,pmed10,200,67,1255,1255.00,0.00,1255,0.00,0.00,1.7720,0.0000,177.00,0.00


,Instance,n,p,Best,BKS,BestGap(\%),MeanGap(\%),Time(s),Mem
0,pmed1,100,5,5819,5819,0.00,0.00,0.0327,127.00
1,pmed2,100,10,4093,4093,0.00,0.00,0.0642,89.00
2,pmed3,100,10,4250,4250,0.00,0.00,0.0560,103.00
3,pmed4,100,20,3034,3034,0.00,0.00,0.1012,102.00
4,pmed5,100,33,1355,1355,0.00,0.00,0.1168,93.00
5,pmed6,200,5,7824,7824,0.00,0.00,0.1868,246.00
6,pmed7,200,10,5631,5631,0.00,0.00,0.3438,135.00
7,pmed8,200,20,4445,4445,0.00,0.00,0.7633,90.00
8,pmed9,200,40,2734,2734,0.00,0.00,0.9859,130.00
9,pmed10,200,67,1255,1255,0.00,0.00,1.7720,177.00


In [18]:
LATEX_CAPTION = 'Benchmark results for the p-median metaheuristic on OR-Library instances'
LATEX_LABEL   = 'tab:pmedian_benchmark'


latex_table_full = publication_full_df.to_latex(
    index        =False,
    escape       =False,
    caption      =LATEX_CAPTION,
    label        =LATEX_LABEL  ,
    column_format='l' + 'r' * (publication_full_df.shape[1] - 1),
)

latex_table_summary = publication_summary_df.to_latex(
    index        =False,
    escape       =False,
    caption      =LATEX_CAPTION + ' (summary)',
    label        =LATEX_LABEL   + '_summary'  ,
    column_format='l' + 'r' * (publication_summary_df.shape[1] - 1),
)

Full publication table:

In [19]:
print(latex_table_full)

\begin{table}
\caption{Benchmark results for the p-median metaheuristic on OR-Library instances}
\label{tab:pmedian_benchmark}
\begin{tabular}{lrrrrrrrrrrrr}
\toprule
Instance & n & p & Best & Mean & Std & BKS & BestGap(\%) & MeanGap(\%) & Time(s) & TimeStd & Mem & MemStd \\
\midrule
pmed1 & 100 & 5 & 5819 & 5819.00 & 0.00 & 5819 & 0.00 & 0.00 & 0.0327 & 0.0000 & 127.00 & 0.00 \\
pmed2 & 100 & 10 & 4093 & 4093.00 & 0.00 & 4093 & 0.00 & 0.00 & 0.0642 & 0.0000 & 89.00 & 0.00 \\
pmed3 & 100 & 10 & 4250 & 4250.00 & 0.00 & 4250 & 0.00 & 0.00 & 0.0560 & 0.0000 & 103.00 & 0.00 \\
pmed4 & 100 & 20 & 3034 & 3034.00 & 0.00 & 3034 & 0.00 & 0.00 & 0.1012 & 0.0000 & 102.00 & 0.00 \\
pmed5 & 100 & 33 & 1355 & 1355.00 & 0.00 & 1355 & 0.00 & 0.00 & 0.1168 & 0.0000 & 93.00 & 0.00 \\
pmed6 & 200 & 5 & 7824 & 7824.00 & 0.00 & 7824 & 0.00 & 0.00 & 0.1868 & 0.0000 & 246.00 & 0.00 \\
pmed7 & 200 & 10 & 5631 & 5631.00 & 0.00 & 5631 & 0.00 & 0.00 & 0.3438 & 0.0000 & 135.00 & 0.00 \\
pmed8 & 200 & 20 & 4445 & 

Summary publication table:

In [20]:
print(latex_table_summary)

\begin{table}
\caption{Benchmark results for the p-median metaheuristic on OR-Library instances (summary)}
\label{tab:pmedian_benchmark_summary}
\begin{tabular}{lrrrrrrrr}
\toprule
Instance & n & p & Best & BKS & BestGap(\%) & MeanGap(\%) & Time(s) & Mem \\
\midrule
pmed1 & 100 & 5 & 5819 & 5819 & 0.00 & 0.00 & 0.0327 & 127.00 \\
pmed2 & 100 & 10 & 4093 & 4093 & 0.00 & 0.00 & 0.0642 & 89.00 \\
pmed3 & 100 & 10 & 4250 & 4250 & 0.00 & 0.00 & 0.0560 & 103.00 \\
pmed4 & 100 & 20 & 3034 & 3034 & 0.00 & 0.00 & 0.1012 & 102.00 \\
pmed5 & 100 & 33 & 1355 & 1355 & 0.00 & 0.00 & 0.1168 & 93.00 \\
pmed6 & 200 & 5 & 7824 & 7824 & 0.00 & 0.00 & 0.1868 & 246.00 \\
pmed7 & 200 & 10 & 5631 & 5631 & 0.00 & 0.00 & 0.3438 & 135.00 \\
pmed8 & 200 & 20 & 4445 & 4445 & 0.00 & 0.00 & 0.7633 & 90.00 \\
pmed9 & 200 & 40 & 2734 & 2734 & 0.00 & 0.00 & 0.9859 & 130.00 \\
pmed10 & 200 & 67 & 1255 & 1255 & 0.00 & 0.00 & 1.7720 & 177.00 \\
pmed11 & 300 & 5 & 7696 & 7696 & 0.00 & 0.00 & 0.6654 & 397.00 \\
pmed12 & 30